In [5]:
"""
tune.py

Hyperparameter tuning for XGBoost with Optuna + MLflow,
using feature_engineered_train/eval, like in 06_hyperparameter_tuning_MLflow.
"""

from pathlib import Path

import joblib
import mlflow
import mlflow.xgboost
import optuna
import pandas as pd
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor


PROC_DIR = Path("data/processed")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def load_fe_data(target: str = "Life Expectancy"):
    train_df = pd.read_csv(PROC_DIR / "feature_engineered_train.csv")
    eval_df = pd.read_csv(PROC_DIR / "feature_engineered_eval.csv")

    X_train = train_df.drop(columns=[target])
    y_train = train_df[target]

    X_eval = eval_df.drop(columns=[target])
    y_eval = eval_df[target]

    return X_train, y_train, X_eval, y_eval


def objective(trial: optuna.Trial) -> float:
    X_train, y_train, X_eval, y_eval = load_fe_data()

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
    }

    with mlflow.start_run(nested=True):
        mlflow.log_params(params)

        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        preds = model.predict(X_eval)
        
        mse = mean_squared_error(y_eval, preds)
        rmse = np.sqrt(mse)
        rmse = mean_squared_error(y_eval, preds)

        mlflow.log_metric("rmse", rmse)

    return rmse


def tune_model(
    study_name: str = "xgb_tuning",
    n_trials: int = 30,
    best_model_path: Path = ARTIFACT_DIR / "best_model_xgb.pkl",
):
    mlflow.set_experiment("xgb_hyperparameter_tuning")

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    print("=== Best trial ===")
    print("Params:", study.best_trial.params)
    print("Best RMSE:", study.best_value)

    # Refit best model on full train
    X_train, y_train, X_eval, y_eval = load_fe_data()
    best_params = study.best_trial.params
    best_params["random_state"] = 42

    best_model = XGBRegressor(**best_params)
    best_model.fit(X_train, y_train)

    joblib.dump(best_model, best_model_path)
    print(f"✅ Saved best tuned model to {best_model_path}")

    return study, best_model


if __name__ == "__main__":
    tune_model()


[I 2025-12-06 19:11:22,543] A new study created in memory with name: xgb_tuning
[I 2025-12-06 19:11:22,680] Trial 0 finished with value: 3.067564219198836 and parameters: {'n_estimators': 493, 'max_depth': 3, 'learning_rate': 0.12104802594035964, 'subsample': 0.8132858210278295, 'colsample_bytree': 0.6120285085514212, 'min_child_weight': 9}. Best is trial 0 with value: 3.067564219198836.
[I 2025-12-06 19:11:22,902] Trial 1 finished with value: 2.720371565674385 and parameters: {'n_estimators': 403, 'max_depth': 4, 'learning_rate': 0.09964718243050211, 'subsample': 0.9395678630390867, 'colsample_bytree': 0.8242701398253909, 'min_child_weight': 1}. Best is trial 1 with value: 2.720371565674385.
[I 2025-12-06 19:11:23,142] Trial 2 finished with value: 2.758636877031672 and parameters: {'n_estimators': 913, 'max_depth': 3, 'learning_rate': 0.03649819794088408, 'subsample': 0.5905942806000773, 'colsample_bytree': 0.7839298912597862, 'min_child_weight': 1}. Best is trial 1 with value: 2.7203

[I 2025-12-06 19:11:30,527] Trial 26 finished with value: 2.746836146822762 and parameters: {'n_estimators': 255, 'max_depth': 5, 'learning_rate': 0.06566605212110677, 'subsample': 0.9248185916170867, 'colsample_bytree': 0.9982282356247871, 'min_child_weight': 1}. Best is trial 21 with value: 2.6001289018484606.
[I 2025-12-06 19:11:30,649] Trial 27 finished with value: 2.6864931648821817 and parameters: {'n_estimators': 332, 'max_depth': 3, 'learning_rate': 0.08752389173204717, 'subsample': 0.9922681838333828, 'colsample_bytree': 0.9433162063026463, 'min_child_weight': 2}. Best is trial 21 with value: 2.6001289018484606.
[I 2025-12-06 19:11:30,934] Trial 28 finished with value: 2.6618737191094666 and parameters: {'n_estimators': 269, 'max_depth': 7, 'learning_rate': 0.0544399486026564, 'subsample': 0.7737584275989994, 'colsample_bytree': 0.8303836270091406, 'min_child_weight': 3}. Best is trial 21 with value: 2.6001289018484606.
[I 2025-12-06 19:11:31,083] Trial 29 finished with value:

=== Best trial ===
Params: {'n_estimators': 204, 'max_depth': 4, 'learning_rate': 0.05001902554227846, 'subsample': 0.9280178330154853, 'colsample_bytree': 0.9318146576321128, 'min_child_weight': 1}
Best RMSE: 2.6001289018484606
✅ Saved best tuned model to artifacts/best_model_xgb.pkl
